# PyTorch MLP Classifier (Tabular)
Train an **MLP** on CSV tabular data with PyTorch. Includes early stopping and evaluation.

In [ ]:

import os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

from src.data_utils import CSVDataset
from src.model import MLP

csv_path = '../data/sample.csv'   # replace with your CSV if desired
feature_cols = [f'f{i}' for i in range(1,11)]
label_col = 'label'
batch_size = 32
lr = 1e-3
max_epochs = 100
patience = 10
val_split = 0.2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


In [ ]:

dataset = CSVDataset(csv_path, feature_cols, label_col)
n_total = len(dataset)
n_val = int(n_total * val_split)
n_train = n_total - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

in_dim = len(feature_cols)
model = MLP(in_dim).to(device)
opt = torch.optim.Adam(model.parameters(), lr=lr)
crit = nn.CrossEntropyLoss()

train_losses, val_losses, val_accs = [], [], []
best_val = float('inf')
best_state = None
no_improve = 0


In [ ]:

# Training loop with early stopping
for epoch in range(1, max_epochs+1):
    model.train()
    ep_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device); yb = yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        loss.backward()
        opt.step()
        ep_loss += loss.item() * len(xb)
    ep_loss /= len(train_loader.dataset)
    train_losses.append(ep_loss)

    # validation
    model.eval()
    val_loss = 0.0
    preds_all, y_all = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device); yb = yb.to(device)
            logits = model(xb)
            loss = crit(logits, yb)
            val_loss += loss.item() * len(xb)
            preds_all.extend(torch.argmax(logits, dim=1).cpu().numpy())
            y_all.extend(yb.cpu().numpy())
    val_loss /= len(val_loader.dataset)
    from sklearn.metrics import accuracy_score
    acc = accuracy_score(y_all, preds_all)
    val_losses.append(val_loss); val_accs.append(acc)

    if val_loss < best_val - 1e-5:
        best_val = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1

    print(f"Epoch {epoch:03d} | train {ep_loss:.4f} | val {val_loss:.4f} | val_acc {acc:.3f} | no_improve {no_improve}")
    if no_improve >= patience:
        print('Early stopping.')
        break

if best_state is not None:
    model.load_state_dict(best_state)


In [ ]:

plt.figure(figsize=(6,4))
plt.plot(train_losses, label='train')
plt.plot(val_losses, label='val')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.title('Loss'); plt.legend(); plt.show()

plt.figure(figsize=(6,4))
plt.plot(val_accs)
plt.xlabel('epoch'); plt.ylabel('val acc'); plt.title('Validation Accuracy'); plt.show()


In [ ]:

from sklearn.metrics import classification_report, confusion_matrix
print('Validation report:')
print(classification_report(y_all, preds_all))
print('Confusion matrix:')
print(confusion_matrix(y_all, preds_all))
